In [33]:
import os
import yaml
import pandas as pd
from pathlib import Path


def _flatten(d: dict, prefix: str = "") -> dict:
    """Recursively flatten a nested dict with dot-separated keys."""
    out = {}
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else k
        print(type(v), key, v)
        if isinstance(v, dict):
            out.update(_flatten(v, key))
        elif isinstance(v, list):
            for i, item in enumerate(v):
                item_key = f"{key}[{i}]"
                if isinstance(item, dict):
                    out.update(_flatten(item, item_key))
                else:
                    out[item_key] = item
        else:
            out[key] = v
    return out


def collect_runs(root: str | os.PathLike) -> pd.DataFrame:
    """
    Walk *root* and return a DataFrame where each row is one run.

    A "run" is any directory that contains a ``params.yaml`` file.

    Columns
    -------
    run_path       : absolute path of the run directory
    <param keys>   : flattened params.yaml entries  (dot-separated for nested keys)
    <metric keys>  : columns from metrics_summary.csv prefixed as metric.class
                     (e.g. ``AP50.all``, ``F1.car``)
    files          : list[str] of filenames present directly in the run directory
    """
    rows = []

    for dirpath, dirnames, filenames in os.walk(root):
        if "params.yaml" not in filenames:
            continue

        run_dir = Path(dirpath)
        row: dict = {"run_path": str(run_dir)}

        # --- params ---
        with open(run_dir / "params.yaml") as f:
            params = yaml.safe_load(f) or {}
        row.update(_flatten(params))

        # --- metrics ---
        metrics_file = run_dir / "metrics_summary.json"

        if metrics_file.exists():
            # Flatten json with dot-separated keys, and prefix with "metric."
            with open(metrics_file) as f:
                metrics = yaml.safe_load(f) or {}
            metrics_flat = _flatten(metrics, prefix="metric")
            row.update(metrics_flat)
        # --- files ---
        row["files"] = sorted(filenames)

        rows.append(row)

    return pd.DataFrame(rows)


In [34]:
OUT_DIR = "../out"

df = collect_runs(OUT_DIR)
df

<class 'list'> classes ['crashed_car', 'person', 'car']
<class 'dict'> model {'name': 'omdetturbo'}
<class 'str'> model.name omdetturbo
<class 'dict'> val {'data': 'data/VistaSynth/data.yaml', 'split': 'test'}
<class 'str'> val.data data/VistaSynth/data.yaml
<class 'str'> val.split test
<class 'dict'> metric.summary {'P': 0.12319203979355912, 'R': 0.18659420289855075, 'mAP50': 0.12499169450541879, 'mAP50-95': 0.055866604674712377, 'fitness': 0.055866604674712377}
<class 'float'> metric.summary.P 0.12319203979355912
<class 'float'> metric.summary.R 0.18659420289855075
<class 'float'> metric.summary.mAP50 0.12499169450541879
<class 'float'> metric.summary.mAP50-95 0.055866604674712377
<class 'float'> metric.summary.fitness 0.055866604674712377
<class 'dict'> metric.speed {'preprocess_ms_per_image': 0.782, 'inference_ms_per_image': 49.329, 'loss_ms_per_image': 0.0, 'postprocess_ms_per_image': 0.001, 'pipeline_ms_per_image': 50.112, 'fps_inference': 20.3, 'fps_pipeline': 20.0, 'total_image

,run_path,classes[0],classes[1],classes[2],model.name,val.data,val.split,metric.summary.P,metric.summary.R,metric.summary.mAP50,...,metric.per_class[2].Images,metric.per_class[2].Instances,metric.per_class[2].Box-P,metric.per_class[2].Box-R,metric.per_class[2].Box-F1,metric.per_class[2].mAP50,metric.per_class[2].mAP50-95,files,model.model,model.model_id
0,../out/2026-04-17_17-17-15_Base/run_0,crashed_car,person,car,omdetturbo,data/VistaSynth/data.yaml,test,0.123192,0.186594,0.124992,...,72.0,256.0,0.00000,0.00000,0.00000,0.00000,0.00000,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....",NaN,NaN
1,../out/2026-04-17_17-04-48_Base/run_3,crashed_car,person,car,yoloe,data/VistaSynth/data.yaml,test,0.778914,0.784364,0.824925,...,72.0,256.0,0.79254,0.78516,0.78883,0.80340,0.47215,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....",checkpoints/YOLOE11_EvaGen.pt,NaN
2,../out/2026-04-17_17-04-48_Base/run_0,crashed_car,person,car,sam,data/VistaSynth/data.yaml,test,0.622826,0.599225,0.557321,...,72.0,256.0,0.41493,0.75352,0.53517,0.40107,0.17465,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....",NaN,NaN
3,../out/2026-04-17_17-04-48_Base/run_1,crashed_car,person,car,moondream,data/VistaSynth/data.yaml,test,0.512157,0.731431,0.503069,...,72.0,256.0,0.41218,0.68750,0.51537,0.41487,0.19092,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....",NaN,NaN
4,../out/2026-04-17_17-04-48_Base/run_2,crashed_car,person,car,moondream,data/VistaSynth/data.yaml,test,0.439841,0.732054,0.439549,...,72.0,256.0,0.37952,0.73828,0.50133,0.41124,0.16981,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....",NaN,vikhyatk/moondream2


In [30]:
list(df.columns)

['run_path',
 'classes',
 'model.name',
 'val.data',
 'val.split',
 'metric.summary.P',
 'metric.summary.R',
 'metric.summary.mAP50',
 'metric.summary.mAP50-95',
 'metric.summary.fitness',
 'metric.speed.preprocess_ms_per_image',
 'metric.speed.inference_ms_per_image',
 'metric.speed.loss_ms_per_image',
 'metric.speed.postprocess_ms_per_image',
 'metric.speed.pipeline_ms_per_image',
 'metric.speed.fps_inference',
 'metric.speed.fps_pipeline',
 'metric.speed.total_images',
 'metric.speed.total_instances',
 'metric.speed.total_pipeline_time_s',
 'metric.per_class',
 'files',
 'model.model',
 'model.model_id']

In [27]:
columns = ["model.name", "model.model_id", 'metric.speed.fps_inference', "files", 'metric.per_class']
df[columns]

,model.name,model.model_id,metric.speed.fps_inference,files,metric.per_class
0,omdetturbo,NaN,20.3,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....","[{'Class': 'crashed_car', 'Images': 100.0, 'In..."
1,yoloe,NaN,210.7,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....","[{'Class': 'crashed car', 'Images': 100.0, 'In..."
2,sam,NaN,1.8,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....","[{'Class': 'crashed_car', 'Images': 100.0, 'In..."
3,moondream,NaN,0.5,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....","[{'Class': 'crashed_car', 'Images': 100.0, 'In..."
4,moondream,vikhyatk/moondream2,2.1,"[BoxF1_curve.png, BoxPR_curve.png, BoxP_curve....","[{'Class': 'crashed_car', 'Images': 100.0, 'In..."


In [ ]:
import json
import yaml
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from pathlib import Path

OUT_DIR = Path("../out")

# ── collect all runs that have a pr_curves.json ───────────────────────────────
runs = []
for pr_path in sorted(OUT_DIR.rglob("pr_curves.json")):
    run_dir = pr_path.parent
    params_file = run_dir / "params.yaml"
    if not params_file.exists():
        continue

    with open(params_file) as f:
        params = yaml.safe_load(f) or {}

    model_name = params.get("model", {}).get("name", run_dir.name)
    model_id   = params.get("model", {}).get("model_id") or ""

    label = f"{model_name} ({model_id})" if model_id else model_name

    with open(pr_path) as f:
        pr_data = json.load(f)

    runs.append({
        "label":    label,
        "recall":   np.array(pr_data["recall"]),
        "per_class": {
            # normalise class names: strip spaces → underscores
            cls.replace(" ", "_"): {
                "precision": np.array(info["precision"]),
                "ap50":      info.get("ap50"),
            }
            for cls, info in pr_data["per_class"].items()
        },
    })

# ── gather all unique class names across every run ────────────────────────────
all_classes = sorted({cls for run in runs for cls in run["per_class"]})

# ── one subplot per class ─────────────────────────────────────────────────────
ncols = min(3, len(all_classes))
nrows = (len(all_classes) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)

colors = cm.tab10(np.linspace(0, 1, len(runs)))

for ax, cls_name in zip(axes.flat, all_classes):
    for run, color in zip(runs, colors):
        if cls_name not in run["per_class"]:
            continue
        info   = run["per_class"][cls_name]
        ap_str = f"  AP50={info['ap50']:.3f}" if info["ap50"] is not None else ""
        ax.plot(run["recall"], info["precision"],
                label=f"{run['label']}{ap_str}", color=color, linewidth=1.5)

    ax.set_title(cls_name.replace("_", " ").title(), fontsize=13)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.3)

# hide unused axes
for ax in axes.flat[len(all_classes):]:
    ax.set_visible(False)

fig.suptitle("PR Curves per Class", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()
